In [3]:
# -*- coding: utf-8 -*-
"""
PCA + 時系列拡張版
------------------

前提:
  - 既に AR + SSDSE のスクリプトを実行済みで、
    panel_ibaraki_ssdse.csv がカレントディレクトリに存在すること。

やること:
  1. panel_ibaraki_ssdse.csv を読み込み、
     year × city_name の行列 (値 = log_pop) を作成
  2. この行列に PCA を適用し、
       - PC1, PC2, PC3 の「年ごとのスコア」（時間軸側のトレンド）
       - 各市町村の「主成分負荷量」（どのPCにどれだけ乗っているか）
     を算出
  3. PCスコアの時系列に AR(1) を当て、2016年以降を予測
  4. 予測したPCスコアから log人口を再構成し、
     実測人口との誤差 (MAPE / RMSE) を評価
  5. 各市町村の主成分負荷量と SSDSE 指標を結合したデータも出力し、
     後で相関・回帰などの要因分析に使えるようにする
"""

import os
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import statsmodels.api as sm


PANEL_FILE = "panel_ibaraki_ssdse.csv"

# 何個の主成分を使うか（とりあえず3つ）
N_COMPONENTS = 3

# 学習に使う最終年（2015年までで学習、2016以降でテスト）
TRAIN_MAX_YEAR = 2015


# ======================================================
# 1. パネルデータの読み込み
# ======================================================
def load_panel(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} が見つかりません。\n"
            "先に AR+SSDSE のスクリプトを実行して "
            "panel_ibaraki_ssdse.csv を作ってください。"
        )
    print("=== Loading panel data from", path, "===")
    df = pd.read_csv(path)
    print("Columns:", list(df.columns)[:20], "...")
    print("Head:")
    print(df.head())
    return df


# ======================================================
# 2. year × city_name の行列を作る
# ======================================================
def build_year_city_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    """
    行: year
    列: city_name
    値: log_pop
    の行列を作る。
    """
    print("\n=== Building year × city matrix (log_pop) ===")
    # ピボット  year × city_name
    mat = panel.pivot_table(
        index="year", columns="city_name", values="log_pop", aggfunc="mean"
    )

    # 年でソート & 列(市町村)でソート
    mat = mat.sort_index().sort_index(axis=1)

    print("Matrix shape (years × cities):", mat.shape)

    # 欠損のある市町村は一旦落とす（PCAのため完全な行列にする）
    n_cities_before = mat.shape[1]
    mat = mat.dropna(axis=1, how="any")
    n_cities_after = mat.shape[1]

    if n_cities_before != n_cities_after:
        print(f"Dropped {n_cities_before - n_cities_after} cities due to missing values.")

    print("Matrix sample:")
    print(mat.head())

    return mat


# ======================================================
# 3. PCA の実行
# ======================================================
def run_pca(logpop_mat: pd.DataFrame, n_components: int = 3):
    """
    log人口行列 (years × cities) に PCA を適用する。

    - rows: years
    - cols: cities

    戻り値:
      - pc_scores_df: 年ごとの主成分スコア (years × n_components)
      - pc_loadings_df: 各市町村の主成分負荷量 (cities × n_components)
      - scaler: StandardScaler (列=市町村 ごとに標準化したもの)
      - pca: PCA オブジェクト
    """
    print("\n=== Running PCA ===")
    X = logpop_mat.values  # shape: (T, N)
    years = logpop_mat.index
    cities = logpop_mat.columns

    # 各市町村列を平均0・分散1に標準化
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_std = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X_std)  # shape: (T, K)
    loadings = pca.components_.T       # shape: (N, K)

    explained = pca.explained_variance_ratio_
    print("Explained variance ratio:", explained)
    print("Cumulative:", np.cumsum(explained))

    pc_scores_df = pd.DataFrame(
        scores,
        index=years,
        columns=[f"PC{k+1}_score" for k in range(n_components)]
    )

    pc_loadings_df = pd.DataFrame(
        loadings,
        index=cities,
        columns=[f"PC{k+1}_loading" for k in range(n_components)]
    )

    print("\nPC scores (time series) sample:")
    print(pc_scores_df.head())
    print("\nPC loadings (by city) sample:")
    print(pc_loadings_df.head())

    # 保存しておく
    pc_scores_df.to_csv("pc_scores_year.csv", encoding="utf-8-sig")
    pc_loadings_df.to_csv("pc_loadings_city.csv", encoding="utf-8-sig")
    print("Saved pc_scores_year.csv and pc_loadings_city.csv")

    return pc_scores_df, pc_loadings_df, scaler, pca


# ======================================================
# 4. PCスコアに AR(1) を当てる
# ======================================================
def fit_ar_to_pc_scores(pc_scores_df: pd.DataFrame, train_max_year: int):
    """
    各主成分スコアの時系列 PCk_score(t) に対して、
      PCk_score(t) = a + b * PCk_score(t-1) + e_t
    を OLS で推定し、2016年以降を予測する。

    戻り値:
      - models: {col_name: statsmodels RegressionResults}
      - pc_scores_pred: pc_scores_df と同じ index・列を持ち、
        2016年以降の部分だけ「予測値」に置き換えた DataFrame
    """
    print("\n=== Fitting AR(1) to PC scores ===")
    pc_scores_df = pc_scores_df.copy()
    years = pc_scores_df.index

    # 予測値を入れていく DataFrame（初期値は実測をコピー）
    pc_scores_pred = pc_scores_df.copy()

    models = {}

    for col in pc_scores_df.columns:
        print(f"\n--- AR(1) for {col} ---")
        df = pc_scores_df[[col]].copy()
        df["lag1"] = df[col].shift(1)
        df["const"] = 1.0

        # 学習データ: train_max_year まで
        train_mask = df.index <= train_max_year
        df_train = df[train_mask].dropna()

        y_train = df_train[col]
        X_train = df_train[["const", "lag1"]]

        model = sm.OLS(y_train, X_train).fit()
        models[col] = model

        print(model.summary())

        # テストデータ: train_max_year より後
        test_mask = df.index > train_max_year
        df_test = df[test_mask].dropna(subset=["lag1"])

        X_test = df_test[["const", "lag1"]]
        y_pred = model.predict(X_test)

        # 予測値で上書き
        pc_scores_pred.loc[df_test.index, col] = y_pred.values

    # 保存しておく（実測と予測のハイブリッド）
    pc_scores_pred.to_csv("pc_scores_year_ar1_pred.csv", encoding="utf-8-sig")
    print("\nSaved pc_scores_year_ar1_pred.csv")

    return models, pc_scores_pred


# ======================================================
# 5. 予測した PC スコアから log人口を再構成し、精度評価
# ======================================================
def reconstruct_logpop_and_evaluate(
    logpop_mat: pd.DataFrame,
    scaler: StandardScaler,
    pca: PCA,
    pc_scores_pred: pd.DataFrame,
    train_max_year: int
):
    """
    予測済みPCスコア (pc_scores_pred) と PCA・標準化情報 (pca, scaler) から、
    log人口行列を再構成し、2016年以降の誤差を評価する。
    """
    print("\n=== Reconstructing log population from predicted PC scores ===")

    years = logpop_mat.index
    cities = logpop_mat.columns

    # PCスコア (T × K)
    S_hat = pc_scores_pred.loc[years].values  # 念のため年順を logpop_mat に合わせる

    # X_std_hat = S_hat @ components  (components: K × N)
    X_std_hat = S_hat @ pca.components_

    # 標準化を元に戻す: X = X_std * scale + mean
    X_hat = X_std_hat * scaler.scale_ + scaler.mean_

    # DataFrame に戻す (log人口)
    logpop_hat = pd.DataFrame(X_hat, index=years, columns=cities)

    # 実測の log人口
    logpop_true = logpop_mat.copy()

    # 2016年以降だけ評価
    test_years = [y for y in years if y > train_max_year]
    log_true_test = logpop_true.loc[test_years].values
    log_pred_test = logpop_hat.loc[test_years].values

    # 人口に戻す
    pop_true_test = np.exp(log_true_test)
    pop_pred_test = np.exp(log_pred_test)

    # 評価指標 (行列をベクトルにフラット化して評価)
    mape = mean_absolute_percentage_error(pop_true_test.ravel(), pop_pred_test.ravel())
    mse = mean_squared_error(pop_true_test.ravel(), pop_pred_test.ravel())
    rmse = np.sqrt(mse)

    print(f"\nPCA+AR(1) model MAPE (test): {mape:.4f}")
    print(f"PCA+AR(1) model RMSE  (test): {rmse:.2f}")

    # 予測結果をロング形式で保存（市町村×年）
    records = []
    for i, year in enumerate(test_years):
        for j, city in enumerate(cities):
            records.append(
                (city, year, pop_true_test[i, j], pop_pred_test[i, j])
            )

    pred_df = pd.DataFrame(
        records,
        columns=["city_name", "year", "pop_true", "pop_pred_pca"]
    )
    pred_df.to_csv("predictions_pca_ar.csv", index=False, encoding="utf-8-sig")
    print("Saved predictions_pca_ar.csv")

    return mape, rmse, logpop_hat


# ======================================================
# 6. 主成分負荷量と SSDSE 指標を結合（要因分析用）
# ======================================================
def build_loadings_with_ssdse(pc_loadings_df: pd.DataFrame, panel: pd.DataFrame):
    """
    各市町村の主成分負荷量 (PC1_loading, PC2_loading, ...) と
    SSDSE の説明変数 (A1301, A1302, ...) を結合した DataFrame を作成する。
    """
    print("\n=== Building loadings + SSDSE table ===")

    # city_name ごとに SSDSEの値を1行にまとめる（最初の年だけ取る）
    ssdse_cols = [c for c in panel.columns if c.startswith("A1") or c.startswith("A4") or c.startswith("A5") or c.startswith("A7")]
    ssdse_cols = [c for c in ssdse_cols if c in panel.columns]

    ssdse_city = (
        panel[["city_name"] + ssdse_cols]
        .drop_duplicates(subset=["city_name"])
        .set_index("city_name")
    )

    # 負荷量とマージ
    loadings_ssdse = pc_loadings_df.join(ssdse_city, how="inner")

    loadings_ssdse.to_csv("pc_loadings_with_ssdse.csv", encoding="utf-8-sig")
    print("Saved pc_loadings_with_ssdse.csv")
    print("Sample:")
    print(loadings_ssdse.head())

    return loadings_ssdse


# ======================================================
# メイン
# ======================================================
def main():
    # 1) パネル読み込み
    panel = load_panel(PANEL_FILE)

    # 2) 年×市町村 行列 (log_pop) を作成
    logpop_mat = build_year_city_matrix(panel)

    # 3) PCA
    pc_scores_df, pc_loadings_df, scaler, pca = run_pca(
        logpop_mat, n_components=N_COMPONENTS
    )

    # 4) PCスコアに AR(1)を当てる
    models, pc_scores_pred = fit_ar_to_pc_scores(pc_scores_df, TRAIN_MAX_YEAR)

    # 5) 予測PCスコアから log人口を再構成して評価
    mape, rmse, logpop_hat = reconstruct_logpop_and_evaluate(
        logpop_mat, scaler, pca, pc_scores_pred, TRAIN_MAX_YEAR
    )

    # 6) 主成分負荷量と SSDSE 指標の結合表を作る（要因分析用）
    loadings_ssdse = build_loadings_with_ssdse(pc_loadings_df, panel)

    print("\n=== DONE ===")
    print(f"PCA+AR(1) MAPE (test): {mape:.4f}")
    print(f"PCA+AR(1) RMSE  (test): {rmse:.2f}")


if __name__ == "__main__":
    main()


=== Loading panel data from panel_ibaraki_ssdse.csv ===
Columns: ['city_name', 'year', 'population', 'log_pop', 'log_pop_lag1', 'city_code', 'A1301', 'A1302', 'A1303', 'A4101', 'A4200', 'A5101', 'A5102', 'A7101', 'A710101'] ...
Head:
  city_name  year  population    log_pop  log_pop_lag1  city_code   A1301  \
0   かすみがうら市  1976     36119.0  10.494574     10.485312        NaN  4376.0   
1   かすみがうら市  1977     36736.0  10.511512     10.494574        NaN  4376.0   
2   かすみがうら市  1978     37387.0  10.529078     10.511512        NaN  4376.0   
3   かすみがうら市  1979     37919.0  10.543208     10.529078        NaN  4376.0   
4   かすみがうら市  1980     38797.0  10.566098     10.543208        NaN  4376.0   

     A1302    A1303  A4101  A4200   A5101   A5102    A7101  A710101  
0  22859.0  12779.0  176.0  594.0  1248.0  1239.0  15271.0  15230.0  
1  22859.0  12779.0  176.0  594.0  1248.0  1239.0  15271.0  15230.0  
2  22859.0  12779.0  176.0  594.0  1248.0  1239.0  15271.0  15230.0  
3  22859.0  12779.0  17

In [4]:
import pandas as pd

df = pd.read_csv("pc_loadings_with_ssdse.csv", index_col="city_name")

# 主成分と SSDSE 指標の相関を見る
corr = df.corr()[["PC1_loading", "PC2_loading", "PC3_loading"]]
print(corr)

# PC2_loading が高い市町村トップ10
print(df[["PC2_loading"]].sort_values("PC2_loading", ascending=False).head(10))


             PC1_loading  PC2_loading  PC3_loading
PC1_loading     1.000000    -0.743467    -0.026613
PC2_loading    -0.743467     1.000000    -0.012423
PC3_loading    -0.026613    -0.012423     1.000000
A1301          -0.548200     0.448647     0.195793
A1302          -0.493293     0.409276     0.191371
A1303          -0.366613     0.306805     0.228495
A4101          -0.544001     0.431939     0.196326
A4200          -0.334836     0.270718     0.208408
A5101          -0.569760     0.472859     0.194690
A5102          -0.544067     0.446126     0.193122
A7101          -0.486330     0.387020     0.207314
A710101        -0.486348     0.386945     0.207465
           PC2_loading
city_name             
ひたちなか市        0.221633
牛久市           0.221231
阿見町           0.221039
鹿嶋市           0.218149
守谷市           0.218130
神栖市           0.216404
土浦市           0.216246
龍ケ崎市          0.215998
水戸市           0.215831
つくば市          0.215255
